In [ ]:
# 📦 Imports
import numpy as np
import pandas as pd
import itertools
from scipy.stats import norm
from scipy.optimize import least_squares, minimize
import statsmodels.api as sm

# 🎲 Step 1: Simulate Distributions
np.random.seed(42)
true_means = np.array([2, 5, 3, 7, 4])
true_vars = np.array([1, 2, 1.5, 2.5, 1.2])
n_distributions = len(true_means)

# 🔁 Step 2: Generate Pairwise Comparisons
pairwise_data = []
bt_data = []

for i, j in itertools.combinations(range(n_distributions), 2):
    samples_i = np.random.normal(true_means[i], np.sqrt(true_vars[i]), 100)
    samples_j = np.random.normal(true_means[j], np.sqrt(true_vars[j]), 100)
    diffs = samples_i - samples_j

    pairwise_data.append({
        'i': i,
        'j': j,
        'mean_diff': np.mean(diffs),
        'var_diff': np.var(diffs)
    })

    wins_i = np.sum(samples_i > samples_j)
    wins_j = 100 - wins_i
    bt_data.append((i, j, wins_i, wins_j))

df = pd.DataFrame(pairwise_data)

# 📊 Step 3: Bradley-Terry Model (Ranking-Based)
X_bt = []
y_bt = []

for i, j, wins_i, wins_j in bt_data:
    for _ in range(wins_i):
        row = [0]*n_distributions
        row[i] = 1
        row[j] = -1
        X_bt.append(row)
        y_bt.append(1)
    for _ in range(wins_j):
        row = [0]*n_distributions
        row[i] = -1
        row[j] = 1
        X_bt.append(row)
        y_bt.append(0)

X_bt = np.array(X_bt)
y_bt = np.array(y_bt)

model_bt = sm.Logit(y_bt, X_bt)
result_bt = model_bt.fit(disp=False)
estimated_bt_means = result_bt.params
print("📈 Bradley-Terry estimated means:")
print(np.round(estimated_bt_means, 3))

# 📐 Step 4: Least Squares for Mean Differences
def residuals(mu):
    return [mu[row['i']] - mu[row['j']] - row['mean_diff'] for row in pairwise_data]

mu0 = np.zeros(n_distributions)
result_ls = least_squares(residuals, mu0)
estimated_ls_means = result_ls.x
print("\n📉 Least Squares estimated means:")
print(np.round(estimated_ls_means, 3))

# 📏 Step 5: Variance Estimation via Optimization
def variance_loss(vars):
    return sum((vars[row['i']] + vars[row['j']] - row['var_diff'])**2 for row in pairwise_data)

var0 = np.ones(n_distributions)
result_var = minimize(variance_loss, var0, bounds=[(0.01, 10)]*n_distributions)
estimated_vars = result_var.x
print("\n📊 Estimated variances:")
print(np.round(estimated_vars, 3))

# 📌 Step 6: Compare with True Values
print("\n✅ True means:", true_means)
print("✅ True variances:", true_vars)
